# HEST-1k Coverage95 Biological Signature Fidelity

This notebook summarizes marker/signature-level biological fidelity for the fixed coverage95 HistoOmniST expression model on held-out HEST human Visium test slides. It reports only outputs generated by `scripts/hest_coverage95_biological_signatures.py`.

In [1]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src" / "histoomnist").exists() and (ROOT.parent / "src" / "histoomnist").exists():
    ROOT = ROOT.parent

out_dir = ROOT / "results/hest1k_human_visium_expression/biological_signatures"
run_summary = json.loads((out_dir / "run_summary.json").read_text(encoding="utf-8"))
signature_coverage = pd.read_csv(out_dir / "signature_gene_coverage.csv")
signature_summary = pd.read_csv(out_dir / "signature_summary.csv")
by_slide = pd.read_csv(out_dir / "signature_by_slide.csv")
by_organ = pd.read_csv(out_dir / "signature_by_organ.csv")
overlay_manifest = pd.read_csv(out_dir / "spatial_signature_map_manifest.csv")

assert run_summary["is_truncated"] is False
assert run_summary["n_spots_evaluated"] == run_summary["n_dataset_spots"] == 121421
assert run_summary["n_signatures"] == len(signature_coverage)
assert overlay_manifest["written"].all()
run_summary

{'expression_config': 'configs/hest1k_human_visium_expression_highconf_symbol95.yaml',
 'sf_config': 'configs/hest1k_human_visium_sf_highconf_context_distribution_light.yaml',
 'expression_checkpoint': 'checkpoints/hest1k_human_visium_expression/highconf_symbol95_rate/best.pt',
 'sf_checkpoint': 'checkpoints/hest1k_human_visium_sf/context_distribution_light_hipt256_leave_slide_out/best.pt',
 'splits': ['test'],
 'n_spots_evaluated': 121421,
 'n_dataset_spots': 121421,
 'n_signatures': 10,
 'min_signature_genes': 2,
 'max_batches': None,
 'is_truncated': False,
 'n_spatial_maps_written': 18,
 'outputs': {'signature_gene_coverage': 'results/hest1k_human_visium_expression/biological_signatures/signature_gene_coverage.csv',
  'signature_summary': 'results/hest1k_human_visium_expression/biological_signatures/signature_summary.csv',
  'signature_by_slide': 'results/hest1k_human_visium_expression/biological_signatures/signature_by_slide.csv',
  'signature_by_organ': 'results/hest1k_human_visi

In [2]:
coverage_view = signature_coverage.sort_values(["n_present_genes", "signature"], ascending=[False, True])
assert (coverage_view["n_present_genes"] >= run_summary["min_signature_genes"]).all()
coverage_view

,signature,n_target_genes,n_present_genes,present_genes,missing_genes
1,pan_immune,8,8,PTPRC|LST1|LYZ|CD68|CD3D|CD3E|CD8A|MS4A1,NaN
5,stromal_ecm,8,8,COL1A1|COL1A2|COL3A1|DCN|LUM|VIM|ACTA2|TAGLN,NaN
9,interferon_response,7,7,ISG15|IFIT1|IFIT3|MX1|OAS1|STAT1|CXCL10,NaN
2,t_cell,7,7,CD3D|CD3E|CD2|CD247|TRAC|CD8A|CD4,NaN
6,endothelial,6,6,PECAM1|VWF|KDR|ENG|PLVAP|RAMP2,NaN
0,epithelial_tumor,8,6,EPCAM|KRT7|KRT8|MUC1|ERBB2|MSLN,KRT18|KRT19
4,myeloid,6,6,LYZ|LST1|CD68|FCGR3A|S100A8|S100A9,NaN
3,b_cell,5,5,MS4A1|CD79A|CD79B|CD74|BANK1,NaN
8,hypoxia_stress,7,5,CA9|ENO1|HIF1A|BNIP3|NDRG1,VEGFA|LDHA
7,proliferation,5,5,MKI67|TOP2A|PCNA|UBE2C|CENPF,NaN


In [3]:
rate = signature_summary[signature_summary["metric_kind"].eq("rate")][
    ["signature", "rate_n_spots", "rate_pearson", "rate_mae", "n_present_genes", "missing_genes"]
].rename(columns={"rate_n_spots": "n_spots", "rate_pearson": "pearson", "rate_mae": "mae"})
count = signature_summary[signature_summary["metric_kind"].eq("count_pred_sf")][
    ["signature", "count_pred_sf_n_spots", "count_pred_sf_pearson", "count_pred_sf_mae"]
].rename(columns={"count_pred_sf_n_spots": "count_n_spots", "count_pred_sf_pearson": "count_pearson", "count_pred_sf_mae": "count_mae"})
combined = rate.merge(count, on="signature", how="inner").sort_values("pearson", ascending=False)
assert len(combined) == run_summary["n_signatures"]
combined

,signature,n_spots,pearson,mae,n_present_genes,missing_genes,count_n_spots,count_pearson,count_mae
2,epithelial_tumor,121421.0,0.866730,0.211455,6,KRT18|KRT19,121421.0,0.852484,0.217978
8,stromal_ecm,117100.0,0.800195,0.532104,8,NaN,117100.0,0.771473,0.529040
5,myeloid,117968.0,0.651851,0.259049,6,NaN,117968.0,0.627516,0.236350
6,pan_immune,117968.0,0.613431,0.121243,8,NaN,117968.0,0.572889,0.115807
0,b_cell,121421.0,0.586945,0.188883,5,NaN,121421.0,0.605659,0.171021
7,proliferation,121421.0,0.580034,0.133362,5,NaN,121421.0,0.607876,0.142305
1,endothelial,117968.0,0.498288,0.210426,6,NaN,117968.0,0.530502,0.195690
3,hypoxia_stress,113647.0,0.496828,0.237531,5,VEGFA|LDHA,113647.0,0.604934,0.236355
4,interferon_response,117968.0,0.490057,0.153695,7,NaN,117968.0,0.567955,0.153756
9,t_cell,117968.0,0.442417,0.069231,7,NaN,117968.0,0.406769,0.067093


In [4]:
organ_rate = by_organ[by_organ["metric_kind"].eq("rate")][
    ["organ", "signature", "rate_n_spots", "rate_pearson", "rate_mae"]
].sort_values(["signature", "rate_pearson"], ascending=[True, False])
assert set(organ_rate["signature"]) == set(combined["signature"])
organ_rate.head(30)

,organ,signature,rate_n_spots,rate_pearson,rate_mae
61,Brain,b_cell,32674.0,0.788904,0.131007
65,Skin,b_cell,12604.0,0.551209,0.192819
62,Breast,b_cell,4321.0,0.421950,0.155333
60,Bowel,b_cell,42772.0,0.394953,0.265156
63,Eye,b_cell,1249.0,0.352549,0.223746
64,Heart,b_cell,27801.0,-0.089020,0.141420
71,Skin,endothelial,12604.0,0.529507,0.153676
66,Bowel,endothelial,42772.0,0.505486,0.256634
69,Eye,endothelial,1249.0,0.373515,0.352999
67,Brain,endothelial,29221.0,0.361817,0.108201


In [5]:
overlay_manifest

,sample_id,signature,path,written
0,MISC33,epithelial_tumor,results/hest1k_human_visium_expression/biologi...,True
1,MISC33,stromal_ecm,results/hest1k_human_visium_expression/biologi...,True
2,MISC33,myeloid,results/hest1k_human_visium_expression/biologi...,True
3,MISC33,pan_immune,results/hest1k_human_visium_expression/biologi...,True
4,MISC33,b_cell,results/hest1k_human_visium_expression/biologi...,True
5,MISC33,proliferation,results/hest1k_human_visium_expression/biologi...,True
6,MISC46,epithelial_tumor,results/hest1k_human_visium_expression/biologi...,True
7,MISC46,stromal_ecm,results/hest1k_human_visium_expression/biologi...,True
8,MISC46,myeloid,results/hest1k_human_visium_expression/biologi...,True
9,MISC46,pan_immune,results/hest1k_human_visium_expression/biologi...,True
